# CH07 - Laboratorio: Listas Enlazadas

Laboratorio del capítulo 7 (Goodrich, Tamassia & Goldwasser).

Completa **todos** los métodos y funciones marcadas con `pass` o `raise NotImplementedError`.
Cada bloque de ejercicio trae sus propias pruebas: ejecútalas para verificar tu solución.

**Contenido**

0. Implementación base de `SinglyLinkedList`
1. `LinkedStack` - pila construida sobre una lista enlazada simple
2. `LinkedQueue` - cola construida sobre una lista enlazada simple
3. Conteo **recursivo** de nodos
4. Iterador (`__iter__`) y `__reversed__`
5. Ejercicios del capítulo (R-7.1, C-7.28, R-7.2, R-7.5, C-7.24, C-7.29, 7.3, 7.11)


---
## 0. Implementación base - `SinglyLinkedList`

Esta es la estructura sobre la que se construye todo el laboratorio. Cada nodo guarda un
**elemento** (`_element`) y una **referencia al siguiente** (`_next`). La lista mantiene
`_head`, `_tail` y `_size`.

| Operación | $O$ |
|---|---|
| `insert_first(e)` | $O(1)$ |
| `insert_last(e)` | $O(1)$ (gracias a `_tail`) |
| `remove_first()` | $O(1)$ |
| `remove_last()` | $O(n)$ |
| acceso por índice | $O(n)$ |

Ejecuta la celda siguiente antes que cualquier otra.

In [1]:
from goodrich.exceptions import Empty


class SinglyLinkedList:
    """Lista enlazada simple con referencias a head y tail."""

    class _Node:
        """Nodo interno: guarda el elemento y la referencia al siguiente."""
        __slots__ = '_element', '_next'
        def __init__(self, element, nxt):
            self._element = element
            self._next    = nxt

    def __init__(self):
        self._head = None
        self._tail = None
        self._size = 0

    def __len__(self):    return self._size
    def is_empty(self):   return self._size == 0

    def first(self):
        if self.is_empty():
            raise Empty('lista vacía')
        return self._head._element

    def last(self):
        if self.is_empty():
            raise Empty('lista vacía')
        return self._tail._element

    def insert_first(self, e):
        self._head = self._Node(e, self._head)
        if self._size == 0:
            self._tail = self._head
        self._size += 1

    def insert_last(self, e):
        node = self._Node(e, None)
        if self.is_empty():
            self._head = node
        else:
            self._tail._next = node
        self._tail = node
        self._size += 1

    def remove_first(self):
        if self.is_empty():
            raise Empty('lista vacía')
        val = self._head._element
        self._head = self._head._next
        self._size -= 1
        if self._size == 0:
            self._tail = None
        return val


# --- Prueba rápida ---
S = SinglyLinkedList()
for x in ['A', 'B', 'C', 'D']:
    S.insert_last(x)
print(len(S), S.first(), S.last())   # 4 A D
S.remove_first()
print(len(S), S.first())             # 3 B

4 A D
3 B


### Recorriendo la lista a mano

Antes de automatizar el recorrido con un iterador, conviene entender cómo se camina la
cadena de nodos empezando en `_head` y saltando con `_next`.

In [2]:
import random
random.seed(10)

L = SinglyLinkedList()
for i in range(500):
    L.insert_first(random.randint(0, 100))

# nodo en la posición 99
current = L._head
for i in range(99):
    current = current._next
print(current._element)

# extraer la lista enlazada como una lista usual de Python
current = L._head
L1 = []
for i in range(len(L)):
    L1.append(current._element)
    current = current._next
print(L1[:10])

58
[36, 27, 74, 51, 73, 12, 60, 97, 81, 65]


---
## 1. `LinkedStack` - pila sobre una lista enlazada simple

Implementa una pila **LIFO** que use una `SinglyLinkedList` como contenedor interno.
La lista se crea en el `__init__` (composición, no herencia): el atributo `_data` es la
única estructura de almacenamiento permitida - no uses `list` ni `deque`.

| Método | Descripción | $O$ esperado |
|---|---|---|
| `push(e)` | Inserta `e` en la cima | $O(1)$ |
| `pop()` | Elimina y retorna la cima. Lanza `Empty` si está vacía | $O(1)$ |
| `top()` | Retorna la cima sin eliminarla. Lanza `Empty` si está vacía | $O(1)$ |
| `is_empty()` | `True` si no hay elementos | $O(1)$ |
| `__len__()` | Número de elementos | $O(1)$ |

**Clave:** ¿por qué la cima debe ser el **frente** de la lista y no el final?

In [6]:
class LinkedStack:
    """Pila (LIFO) implementada con una SinglyLinkedList como contenedor."""

    def __init__(self):
        self._data = SinglyLinkedList()

    def __len__(self):
        return len(self._data)

    def is_empty(self):
        return self._data.is_empty()

    def push(self, e):
        """Inserta el elemento e en la cima de la pila."""
        self._data.insert_first(e)

    def top(self):
        """Retorna (sin eliminar) el elemento en la cima. Lanza Empty si está vacía."""
        if self.is_empty():
            return Empty
        return self._data.first()

    def pop(self):
        """Elimina y retorna el elemento en la cima. Lanza Empty si está vacía."""
        if self.is_empty():
            return Empty
        return self._data.remove_first()

In [7]:
# --- Tests ---
P = LinkedStack()
print(P.is_empty())               # True
P.push(1); P.push(2); P.push(3)
print(len(P), P.top())            # 3 3
print(P.pop(), P.pop())           # 3 2
print(len(P), P.is_empty())       # 1 False
print(P.pop())                    # 1
try:
    P.pop()
except Empty as e:
    print('Empty:', e)

True
3 3
3 2
1 False
1


---
## 2. `LinkedQueue` - cola sobre una lista enlazada simple

Ahora una cola **FIFO**, también con una `SinglyLinkedList` creada en el `__init__`.

| Método | Descripción | $O$ esperado |
|---|---|---|
| `enqueue(e)` | Encola `e` al final | $O(1)$ |
| `dequeue()` | Elimina y retorna el frente. Lanza `Empty` si está vacía | $O(1)$ |
| `first()` | Retorna el frente sin eliminarlo | $O(1)$ |
| `is_empty()` / `__len__()` | Estado de la cola | $O(1)$ |

**Clave:** para que ambas operaciones sean $O(1)$, ¿en qué extremo debe encolar y en cuál
desencolar? Recuerda que eliminar el último nodo de una lista simple es $O(n)$.

In [8]:
class LinkedQueue:
    """Cola (FIFO) implementada con una SinglyLinkedList como contenedor."""

    def __init__(self):
        self._data = SinglyLinkedList()

    def __len__(self):
        return len(self._data)

    def is_empty(self):
        return self._data.is_empty()

    def enqueue(self, e):
        """Inserta el elemento e al final de la cola."""
        self._data.insert_last(e)

    def first(self):
        """Retorna (sin eliminar) el elemento al frente. Lanza Empty si está vacía."""
        if self.is_empty():
            return Empty
        return self._data.first()

    def dequeue(self):
        """Elimina y retorna el elemento al frente. Lanza Empty si está vacía."""
        if self.is_empty():
            return Empty
        return self._data.remove_first()

In [ ]:
# --- Tests ---
Q = LinkedQueue()
print(Q.is_empty())                     # True
for x in ['a', 'b', 'c']:
    Q.enqueue(x)
print(len(Q), Q.first())                # 3 a
print(Q.dequeue(), Q.dequeue())         # a b
Q.enqueue('d')
print(len(Q), Q.first())                # 2 c
print(Q.dequeue(), Q.dequeue())         # c d
try:
    Q.dequeue()
except Empty as e:
    print('Empty:', e)

True
3 a
a b
2 c
c d


---
## 3. Conteo recursivo de nodos *(R-7.3)*

> *Describe a recursive algorithm that counts the number of nodes in a singly linked list.*

**Idea.** El conteo se define sobre el **nodo**, no sobre la lista:

$$
\text{contar}(nodo)=
\begin{cases}
0 & \text{si } nodo = \texttt{None} \quad \text{(caso base)}\\[4pt]
1 + \text{contar}(nodo.\_next) & \text{en otro caso}
\end{cases}
$$

- **Caso base:** una cadena vacía (`None`) tiene 0 nodos.
- **Paso recursivo:** el nodo actual aporta 1, más lo que aporte el resto de la cadena.
- **Complejidad:** $O(n)$ en tiempo y $O(n)$ en espacio (profundidad de la pila de
  llamadas). Con listas muy largas Python lanza `RecursionError`.

Implementa `contar_nodos(nodo)` (recibe un nodo) y `contar_recursivo(L)`
(recibe la lista y arranca la recursión en `L._head`).

In [10]:
def contar_nodos(nodo):
    """Cuenta recursivamente los nodos de la cadena que empieza en `nodo`."""
    if nodo is None:            # caso base: cadena vacía
        return 0
    return 1 + contar_nodos(nodo._next)   # 1 (nodo actual) + resto de la cadena


def contar_recursivo(L):
    """Cuenta recursivamente los nodos de la SinglyLinkedList L, sin usar len(L)."""
    return contar_nodos(L._head)


# --- Tests ---
vacía = SinglyLinkedList()
print(contar_recursivo(vacía))            # 0

C = SinglyLinkedList()
for i in range(7):
    C.insert_last(i)
print(contar_recursivo(C))                # 7
print(contar_recursivo(C) == len(C))      # True

0
7
True


**Para pensar:** ¿a partir de cuántos nodos falla la versión recursiva en tu máquina?
Compárala con una versión iterativa y explica la diferencia en uso de memoria.

In [11]:
# Opcional: encuentra el límite práctico de la recursión
import sys
print('límite de recursión:', sys.getrecursionlimit())

G = SinglyLinkedList()
for i in range(5000):
    G.insert_first(i)

try:
    print(contar_recursivo(G))
except RecursionError as e:
    print('RecursionError:', e)

límite de recursión: 1000
RecursionError: maximum recursion depth exceeded


---
## 4. Iterador y `__reversed__` en `SinglyLinkedList`

Implementa en la subclase `SinglyLinkedList1`:

- **`__iter__`**: recorre los elementos **del frente al final**. Escríbelo como *generador*
  (con `yield`): parte de `_head` y avanza con `_next`. Costo $O(n)$, espacio $O(1)$.
- **`__reversed__`**: recorre los elementos **del final al frente**. En una lista *simple*
  no hay puntero `_prev`, así que necesitas una estrategia: acumular los elementos y
  devolverlos al revés, o recursión. Explica en un comentario qué costo tiene tu solución.

Al tener `__iter__`, la clase funciona automáticamente con `for`, `list()`, `max()`,
`min()`, `sum()`, `in`, desempaquetado, etc.

In [14]:
class SinglyLinkedList1(SinglyLinkedList):
    """SinglyLinkedList con soporte de iteración hacia adelante y hacia atrás."""

    def __iter__(self):
        """Genera los elementos desde el primero hasta el último."""
        cursor = self._head
        while cursor is not None:
            yield cursor._element
            cursor = cursor._next

    def __reversed__(self):
        """Genera los elementos desde el último hasta el primero."""
        elementos = list(self)          # usa __iter__ -> O(n)
        for x in reversed(elementos):   # recorre la lista auxiliar al revés
            yield x

In [15]:
# --- Tests ---
S = SinglyLinkedList1()
for i in range(10):
    S.insert_last(i)

print([x for x in S])                 # [0, 1, ..., 9]
print(list(reversed(S)))              # [9, 8, ..., 0]

it = iter(S)
print(next(it), next(it))             # 0 1

print(max(S), min(S), sum(S))         # 9 0 45
print(7 in S, 99 in S)                # True False

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
0 1
9 0 45
True False


---
# 5. Ejercicios del capítulo

Ejercicios seleccionados de Goodrich, Tamassia & Goldwasser - *Data Structures and
Algorithms in Python*, Capítulo 7.

### Ejercicio 1 - Penúltimo nodo *(R-7.1)*

Implementa `penúltimo(L)` que reciba una `SinglyLinkedList` y retorne el **elemento** del penúltimo nodo (el nodo justo antes del último). Debe lanzar `ValueError` si la lista tiene menos de 2 elementos.

**Restricción:** usa solo `_head` y `_next`; no uses `len(L)` ni `_tail`.

In [16]:
def penúltimo(L):
    """Retorna el elemento del penúltimo nodo de la lista enlazada simple L."""
    if L._head is None or L._head._next is None:
        raise ValueError('lista con menos de 2 elementos')
    cursor = L._head
    while cursor._next._next is not None:   # se detiene cuando cursor._next es el último
        cursor = cursor._next
    return cursor._element


# --- Tests ---
S = SinglyLinkedList()
for x in ['A', 'B', 'C', 'D']:
    S.insert_last(x)
print(penúltimo(S))   # C

S2 = SinglyLinkedList()
S2.insert_last(42)
try:
    penúltimo(S2)
except ValueError as e:
    print(e)           # lista con menos de 2 elementos

C
lista con menos de 2 elementos


### Ejercicio 2 - Revertir una lista enlazada simple *(C-7.28)*

Implementa `revertir(L)` que invierta **en sitio** una `SinglyLinkedList` en $O(n)$ tiempo y $O(1)$ espacio adicional. No crees nuevos nodos; solo redirige los punteros `_next`.

**Pista:** necesitas tres referencias: `prev`, `curr` y `siguiente`. Al terminar, `L._head` debe apuntar al antiguo último nodo (y `L._tail` al antiguo primero).

In [ ]:
def revertir(L):
    """Invierte en sitio la SinglyLinkedList L. No crea nodos nuevos."""
    raise NotImplementedError


# --- Tests ---
R = SinglyLinkedList()
for x in [1, 2, 3, 4, 5]:
    R.insert_last(x)

revertir(R)

nodo = R._head
while nodo is not None:
    print(nodo._element, end=' ')
    nodo = nodo._next
# Esperado: 5 4 3 2 1

### Ejercicio 3 - Concatenar dos listas en $O(1)$ *(R-7.2)*

Implementa `concatenar(L1, L2)` que una todos los nodos de `L2` al final de `L1` en **tiempo constante** $O(1)$ - sin recorrer los nodos. Después de la operación `L2` debe quedar vacía.

**Pista:** conecta `L1._tail._next = L2._head`, actualiza `L1._tail` y `L1._size`. No olvides el caso en que alguna de las dos listas esté vacía.

In [ ]:
def concatenar(L1, L2):
    """Concatena L2 al final de L1 en O(1). L2 queda vacía."""
    raise NotImplementedError


# --- Tests ---
A = SinglyLinkedList()
for x in ['a', 'b', 'c']:
    A.insert_last(x)

B = SinglyLinkedList()
for x in ['d', 'e']:
    B.insert_last(x)

concatenar(A, B)

print(len(A), len(B))    # 5  0
nodo = A._head
while nodo is not None:
    print(nodo._element, end=' ')
    nodo = nodo._next
# Esperado: a b c d e

### Ejercicio 4 - Ejercicio 7.3 sobre `SinglyLinkedList`

Trabaja sobre una lista enlazada simple pequena y resuelve las variantes `A` y `B`
planteadas en clase. Documenta con un comentario qué hace cada versión y su complejidad.

In [ ]:
S = SinglyLinkedList()
for i in range(3):
    S.insert_last(i)


def ejercicio_7_3(S):
    pass


def ejercicio_7_3A(S: SinglyLinkedList):
    pass


def ejercicio_7_3B(S: SinglyLinkedList):
    pass


print(ejercicio_7_3(S))
print(ejercicio_7_3A(S))
print(ejercicio_7_3B(S))

### Ejercicio 5 - Ejercicio 7.11 con `SinglyLinkedList`

Resuelve el ejercicio 7.11 usando una lista enlazada simple como estructura de trabajo.

In [ ]:
def ejercicio_7_11(S: SinglyLinkedList):
    pass


S = SinglyLinkedList()
for i in range(10):
    S.insert_last(i)
print(ejercicio_7_11(S))

### Ejercicio 6 - Pila con lista enlazada simple y nodo centinela *(C-7.24)*

Implementa la clase `LinkedStackCentinela` que use internamente una **lista enlazada simple con un nodo centinela `_header`** (sin dato útil) como almacenamiento. Debe soportar:

| Método | Descripción |
|---|---|
| `push(e)` | Inserta `e` al frente (después del centinela) |
| `pop()` | Elimina y retorna el elemento en la cima |
| `top()` | Retorna (sin eliminar) el elemento en la cima |
| `is_empty()` | `True` si la pila está vacía |
| `__len__()` | Número de elementos |

**Restricción:** el almacenamiento interno solo puede ser la lista con centinela - no uses `list` ni `deque`.

**Para pensar:** compara este diseño con el `LinkedStack` de la sección 1. ¿Qué casos borde
desaparecen gracias al centinela?

In [ ]:
class LinkedStackCentinela:
    """Pila LIFO sobre lista enlazada simple con nodo centinela."""

    class _Node:
        __slots__ = '_element', '_next'
        def __init__(self, e, n): self._element = e; self._next = n

    def __init__(self):
        self._header = self._Node(None, None)   # centinela
        self._size   = 0

    def __len__(self):
        raise NotImplementedError

    def is_empty(self):
        raise NotImplementedError

    def push(self, e):
        raise NotImplementedError

    def top(self):
        raise NotImplementedError

    def pop(self):
        raise NotImplementedError


# --- Tests ---
ls = LinkedStackCentinela()
for v in [1, 2, 3, 4]:
    ls.push(v)

print(len(ls), ls.top())   # 4  4
print(ls.pop())            # 4
print(ls.pop())            # 3
print(len(ls))             # 2

### Ejercicios sobre lista circular y doblemente enlazada

Las siguientes celdas usan las implementaciones del repositorio del curso
(`CircularQueue` y `LinkedDeque`). Ejecuta primero la celda de importacion.

In [ ]:
# `RAIZ` ya fue agregada a sys.path en la celda de la sección 0
from goodrich.ch07.circular_queue import *
from goodrich.ch07.linked_deque import *

### Ejercicio 7 - Recuento en lista circular *(R-7.5)*

Implementa `longitud_circular(cola)` que, dada una `CircularQueue`, cuente y retorne el número de nodos **sin modificar la cola** y **sin usar `len()`**. Utiliza solo `cola._tail` y el acceso a `_next`.

**Pista:** comienza en `_tail._next` (el frente) y avanza hasta volver a `_tail`.

In [ ]:
def longitud_circular(cola):
    """Cuenta los nodos de CircularQueue sin usar len() ni modificar la cola."""
    raise NotImplementedError


# --- Tests ---
cq = CircularQueue()
print(longitud_circular(cq))   # 0  (cola vacía)

for v in [10, 20, 30, 40]:
    cq.enqueue(v)
print(longitud_circular(cq))   # 4
cq.dequeue()
print(longitud_circular(cq))   # 3

### Ejercicio 8 - Revertir una lista doblemente enlazada *(C-7.29 - variante)*

Implementa `revertir_doble(deque)` que invierta en sitio un `LinkedDeque` en $O(n)$ y $O(1)$ espacio adicional. No crees nodos nuevos; solo intercambia los punteros `_prev` y `_next` de cada nodo, y actualiza los centinelas.

**Pista:** recorre los nodos intercambiando `_prev` y `_next` en cada uno; al terminar intercambia también `_header._next` con `_trailer._prev`.

In [ ]:
def revertir_doble(deque):
    """Invierte en sitio el LinkedDeque. No crea nodos nuevos."""
    raise NotImplementedError


# --- Tests ---
d = LinkedDeque()
for x in [1, 2, 3, 4, 5]:
    d.insert_last(x)

revertir_doble(d)

print(d.first(), d.last(), len(d))   # 5  1  5

elementos = []
nodo = d._header._next
while nodo is not d._trailer:
    elementos.append(nodo._element)
    nodo = nodo._next
print(elementos)   # [5, 4, 3, 2, 1]